In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import diags
from scipy.sparse.linalg import factorized
import ipywidgets as widg
from IPython.display import display


# Chen2020 constants (graphite negative electrode, LG M50)

F     = 96485.0
R_gas = 8.314
c_e   = 1000.0       # electrolyte concentration [mol/m³]
c_max = 33133.0      # max solid concentration [mol/m³]  Chen2020
m_ref = 6.48e-7      # BV rate constant [(A/m²)(m³/mol)^1.5]  Chen2020
E_r   = 35000.0      # Arrhenius activation energy [J/mol]  Chen2020
T_ref = 298.15       # reference temperature [K]

#Arrhenius-corrected i0 (Chen2020 )
# i0 = m_ref · exp(E_r/R·(1/T_ref − 1/T)) · √(c_e · c_s · (c_max−c_s)) Temperature slider scales i₀ via Arrhenius
# The exp() term = 1 at T=T_ref, >1 when hot, <1 when cold.
def i0_func(c_s, T, k=m_ref):
    c_s = np.clip(c_s, 1e-12, c_max - 1e-12)
    arrhenius = np.exp(E_r / R_gas * (1 / T_ref - 1 / T))
    return k * arrhenius * np.sqrt(c_e * c_s * (c_max - c_s))

#Graphite OCP(Open Circuit Potential) vs stoichiometry (Chen2020 )
# sto = c_s / c_max  (varies from 0 to 1)
def ocp_graphite(sto):
    return (
        1.9793 * np.exp(-39.3631 * sto)
        + 0.2482
        - 0.0909  * np.tanh(29.8538 * (sto - 0.1234))
        - 0.04478 * np.tanh(14.9159 * (sto - 0.2769))
        - 0.0205  * np.tanh(30.4444 * (sto - 0.6103))
    )


# Core simulation
def run_simulation(C_rate, T, SOC_init, SOC_max, eta_max, a_um, D_exp, k):
    a      = a_um * 1e-6
    D_val  = 10.0 ** D_exp
    c_init = SOC_init * c_max / 100.0  #initial C =initial SOC entered by slider* c_max/100
    j_app  = -(a * c_max * C_rate) / (3.0 * 3600.0)   # negative = insertion
    #j= total flux = total moles/SA*time=(4/3pi*a^3*c_max/4pi*a^2*tau)=(a*c_max/3*tau) where tau = 3600/C_rate
    # i0 with current T and k from sliders
    def i0(c_s):
        return i0_func(c_s, T, k)

    # Inverse BV: j → η  (η = deviation from OCP)
    def eta_from_j(c_s, j):
        i0v = i0(c_s)
        if i0v < 1e-16:
            return np.sign(j) * 10.0 # large default
        return (2 * R_gas * T / F) * np.arcsinh(F * j / (2 * i0v))

    # Mesh (u = C·r)
    nx = 50
    dr = a / nx
    r  = (np.arange(nx) + 0.5) * dr #if np=5 -> np.arrange gives [1,2,3,4]. add 0.5 to get to cell centre.So we get mesh centre list
    dV = r**2

    # Constant diffusion matrix
    dt  = 10.0
    lam = D_val / dr**2
    c_  = lam * dt
    main  = np.full(nx, 1 + 2.0*c_)
    lower = np.full(nx - 1, -c_)
    upper = np.full(nx - 1, -c_)
    main[0]  = 1;  upper[0] = 0.0          # Dirichlet at r=0
    main[-1] = 1 + c_ * (1 - dr / a)    # Neumann at r=a
    solve_A  = factorized(diags([lower, main, upper], [-1, 0, 1], format='csr'))

    u         = c_init * r
    sim_time  = 3600.0
    steps     = int(sim_time / dt)
    snap_t    = [0, 600, 1200, 1800, 3600] #snapshot times
    snap_s    = {int(t / dt) for t in snap_t if t <= sim_time} #snapshot step no
    snapshots = {}

    log_n      = steps + 1 #Number of Logged Time Points(including initial state at t=0)
    time_log   = np.linspace(0, sim_time, log_n) #Creates evenly spaced times:[0, 10, 20, ..., 3600]
    c_surf_log = np.empty(log_n)
    soc_log    = np.empty(log_n)
    eta_log    = np.empty(log_n)
    ocp_log    = np.empty(log_n)

    stop_step   = steps #max possible steps. Either all steps completed or stop loop when SOC or eta limit reached
    stop_reason = "Max time (3600 s)"

    for step in range(steps + 1):
        c_surf = np.clip(u[-1] / a, 1e-12, c_max - 1e-12) #Bound c value on particle surface
        eta    = eta_from_j(c_surf, j_app)

        rhs     = u.copy()
        rhs[0]  = 0.0 #Left BC
        rhs[-1] -= dt * a / dr * j_app #Right BC
        u = solve_A(rhs)

        c_surf_new       = np.clip(u[-1] / a, 1e-12, c_max - 1e-12)
        c_surf_log[step] = c_surf_new
        eta_log[step]    = eta
        ocp_log[step]    = ocp_graphite(c_surf_new / c_max)

        C_profile    = u / r
        C_profile[0] = (4*C_profile[1] - C_profile[2]) / 3 #Replace first node(at r=dr/2) value with centre value using 2nd order interpolation to avoid singularity
        soc_log[step] = np.dot(C_profile, dV) / dV.sum() / c_max #SOC=sigma(C*r^2)/sigma(r**2)

        if step in snap_s:
            snapshots[step * dt] = C_profile.copy() #Dictionary that stores conc profiles frozen at different timestamps

        if soc_log[step] >= SOC_max:
            stop_reason = f"SoC limit ({SOC_max*100:.0f}%) at t={step*dt:.0f} s"
            stop_step = step;  break
        if abs(eta_log[step]) > eta_max:
            stop_reason = f"η limit ({eta_max*1000:.0f} mV) at t={step*dt:.0f} s"
            stop_step = step;  break

    sl = stop_step + 1 #to include the stopping point itself
    return dict(time=time_log[:sl], c_surf=c_surf_log[:sl], soc=soc_log[:sl],
                eta=eta_log[:sl], ocp=ocp_log[:sl], snaps=snapshots,
                r=r, i0=i0, reason=stop_reason, snap_t=snap_t) #returns {t, c(r,t), SOC(t), η(t), OCP(t)}


#widg

style  = {"description_width": "160px"}
layout =widg.Layout(width="420px")

w_C_rate  =widg.FloatSlider(value=1,   min=0.1,   max=5.0,  step=0.1,
                                description="C-rate (C)", style=style, layout=layout, continuous_update=True)
w_T      =widg.FloatSlider(value=298.15, min=263.15, max=333.15, step=5.0,
                                description="Temperature (K)", readout_format=".2f",
                                style=style, layout=layout, continuous_update=True)
w_c_init  =widg.FloatSlider(value=3.79,  min=1,   max=80.0, step=1,
                                description="Initial SoC (%)", style=style, layout=layout, continuous_update=True)
w_SOCmax =widg.FloatSlider(value=0.95,  min=0.80,  max=10, step=0.01,
                                description="SoC cut-off", readout_format=".2f",
                                style=style, layout=layout, continuous_update=True)
w_etamax =widg.FloatSlider(value=0.5,   min=0.05,  max=1,  step=0.05,
                                description="η cut-off (V)", readout_format=".2f",
                                style=style, layout=layout, continuous_update=True)
w_a      =widg.FloatSlider(value=5.0,   min=1,   max=20.0, step=0.5,
                                description="Particle radius (µm)", style=style, layout=layout, continuous_update=True)
w_D_exp   =widg.FloatSlider(value=-13.48, min=-15.0, max=-12.0, step=0.1,
                                description="log₁₀(D) [m²/s]", readout_format=".1f",
                                style=style, layout=layout, continuous_update=True)
w_k      =widg.FloatLogSlider(value=6.48e-7, base=10, min=-8, max=-4, step=0.1,
                                   description="k (rate const.)", readout_format=".2e",
                                   style=style, layout=layout, continuous_update=True)


# Update function (called on every slider change)

def update(C_rate, T, SOC_init, SOC_max, eta_max, a_um, D_exp, k):
    res    = run_simulation(C_rate, T, SOC_init, SOC_max, eta_max, a_um, D_exp, k)
    t      = res["time"];     c_surf = res["c_surf"]
    soc    = res["soc"];      eta    = res["eta"]
    ocp    = res["ocp"];      r      = res["r"]
    snaps  = res["snaps"];    i0     = res["i0"]
    snap_t = res["snap_t"]
#Every time a slider changes, this function- runs the battery simulation,extract results updates all plots

    arrhenius = np.exp(E_r / R_gas * (1 / T_ref - 1 / T))

    fig, axes = plt.subplots(2, 3, figsize=(12, 7))
    fig.suptitle(
        f"{res['reason']}  |  SoC: {soc[-1]*100:.1f}%  |  "
        f"η: {eta[-1]*1000:.1f} mV  |  "
        f"Arrhenius: {arrhenius:.3f}  (i0 × {arrhenius:.2f} vs 25°C)",
        fontsize=10, fontweight="bold"
    )

    # 1. Concentration profiles
    ax = axes[0, 0]
    for col, ts in zip(plt.cm.viridis(np.linspace(0,1,len(snap_t))), snap_t):
        if ts in snaps:
            ax.plot(r*1e6, snaps[ts], lw=2, label=f'{ts:.0f} s', color=col)
    ax.set(xlabel='r (µm)', ylabel='C (mol/m³)', title='Concentration Profiles')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    # 2. Surface concentration
    ax = axes[0, 1]
    ax.plot(t, c_surf, 'b-', lw=2)
    ax.axhline(c_max, color='r', ls='--', alpha=0.6, label='c_max')
    ax.set(xlabel='Time (s)', ylabel='Surface Conc. (mol/m³)', title='Surface Concentration')
    ax.legend(); ax.grid(alpha=0.3)

    # 3. Overpotential
    ax = axes[0, 2]
    ax.plot(t, eta*1000, 'r-', lw=2, label='η')
    ax.axhline( eta_max*1000, color='k', ls='--', alpha=0.5, label=f'±{eta_max*1000:.0f} mV limit')
    ax.axhline(-eta_max*1000, color='k', ls='--', alpha=0.5)
    ax.set(xlabel='Time (s)', ylabel='Overpotential (mV)', title='Overpotential')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    # 4. SoC
    ax = axes[1, 0]
    ax.plot(t, soc*100, 'm-', lw=2)
    ax.axhline(SOC_max*100, color='k', ls='--', alpha=0.5, label=f'cut-off {SOC_max*100:.0f}%')
    ax.set(xlabel='Time (s)', ylabel='SoC (%)', title='State of Charge')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    # 5. OCP + i0 vs SoC  ← NEW: shows Chen2020 dependencies
    ax = axes[1, 1]
    ax.plot(soc*100, ocp, 'g-', lw=2, label='OCP U(sto)')
    ax.set(xlabel='SoC (%)', ylabel='OCP (V)', title='OCP & i₀ vs SoC (Chen2020)')
    ax.legend(loc='upper left', fontsize=8); ax.grid(alpha=0.3)
    ax2 = ax.twinx()
    ax2.plot(soc*100, i0(c_surf), 'c--', lw=1.5, alpha=0.8, label='i₀ (A/m²)')
    ax2.set_ylabel('i₀ (A/m²)', color='c')
    ax2.legend(loc='upper right', fontsize=8)

    # 6. Surface occupancy
    ax = axes[1, 2]
    ax.plot(t, c_surf/c_max*100, 'b-', lw=2)
    ax.axhline(SOC_max*100, color='k', ls='--', alpha=0.5, label=f'cut-off {SOC_max*100:.0f}%')
    ax.set(xlabel='Time (s)', ylabel='Surface Occupancy (%)', title='Surface Filling')
    ax.set_ylim(0, 105); ax.legend(fontsize=8); ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

# ── Wire up and display ────────────────────────────────────────────
out =widg.interactive_output(update, {
    "C_rate": w_C_rate, "T": w_T, "SOC_init": w_c_init,
    "SOC_max": w_SOCmax, "eta_max": w_etamax,
    "a_um": w_a, "D_exp": w_D_exp, "k": w_k,
})

col1 =widg.VBox([widg.HTML("<b>Charging Conditions</b>"),
                     w_C_rate, w_T, w_c_init, w_SOCmax, w_etamax])
col2 =widg.VBox([widg.HTML("<b>Material Parameters</b>"),
                     w_a, w_D_exp, w_k])
note =widg.HTML(
    "<small><b>Note:</b>  "
    "exp(E<sub>r</sub>/R·(1/T<sub>ref</sub>−1/T)) with E<sub>r</sub>=35 kJ/mol (Chen2020). "
    "OCP panel shows graphite equilibrium potential curve.</small>"
)

display(widg.VBox([
   widg.HTML("<h3 style='margin-bottom:4px'>⚡ SPM CC Charging — Live Simulation (Chen2020 parameters)</h3>"),
    note,
   widg.HBox([col1,widg.HTML("&nbsp;"*4), col2],
                 layout=widg.Layout(align_items="flex-start")),
    out,
]))
